# 09 - Foundry IQ knowledge base: existing indexes + Web IQ

## Required permissions

| Identity | Permission | Function |
|---|---|---|
| Signed-in user | **Search Service Contributor** on Search | Create knowledge sources and the knowledge base |
| Signed-in user | **Search Index Data Reader** (or **Search Index Data Contributor**) on Search, plus access to the indexed SharePoint documents | Retrieve indexed content with SharePoint permissions enforced |
| Source author | Web IQ profile API key | Store the key in this knowledge source for notebook and portal retrieval |
| Search managed identity | **Cognitive Services User on the parent Foundry resource** | Call the chat model and existing index vectorizers |

**Resource names:** keep the same `WORKSHOP_PREFIX` used in earlier notebooks. This notebook creates `{prefix}-kb-blob`, `{prefix}-kb-sharepoint`, `{prefix}-kb-webiq` and `{prefix}-kb`. Notebooks 10 and 12 reuse these names. Prefixes prevent naming collisions, not enforce access isolation.

Build **one Foundry IQ knowledge base** over the content already indexed in earlier notebooks, plus live Web IQ:

| Knowledge source | Kind | Existing content |
|---|---|---|
| Blob documents | `searchIndex` | Notebook 04's `{prefix}-blob-chunks` (or notebook 05's `{prefix}-cu-chunks`) |
| SharePoint documents | `searchIndex` | Notebook 07's `{prefix}-nb7-index`, including its Entra user/group ACLs |
| Web IQ | `mcpServer` | Live public web through `https://api.microsoft.ai/v3/mcp` |

The two `searchIndex` knowledge sources are **references to existing indexes**. This notebook does not upload documents, create indexes, change index schemas, or create/run data sources, skillsets or indexers. It does not use remote SharePoint or depend on notebook 08.

## Prerequisites

- Complete notebook **04** (or **05**) and **07** using the same Search service and `WORKSHOP_PREFIX`. Their indexes must already contain content you can retrieve. Indexing and SharePoint ingestion-app setup stay in those notebooks.
- Keep the deployment settings in `.env.azure` and `WORKSHOP_PREFIX` plus `WEBIQ_API_KEY` in `.env`. Sign in with `az login` in the deployment tenant.
- The Search managed identity must be able to call the deployed chat model and the existing indexes' embedding vectorizers. Bicep grants **Cognitive Services User on the parent Foundry resource**.
- No Blob Storage access, SharePoint ingestion-app credentials, or Microsoft 365 Copilot license is needed to run this notebook. SharePoint document access is still required; an existing index does not bypass its ACLs.

These are preview APIs (`2026-08-01-preview`). Retrieval and Web IQ requests can incur charges, but this notebook starts no indexing work.

In [1]:
import json
import os
import re
from pathlib import Path
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_AI_ACCOUNT_NAME',
            'AZURE_AI_MODEL_DEPLOYMENT_NAME', 'WEBIQ_API_KEY')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. See .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Resource prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
aoai_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
chat_deployment = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
chat_model = os.getenv('AZURE_AI_MODEL_NAME', 'gpt-5-mini')
webiq_api_key = os.environ['WEBIQ_API_KEY']
webiq_tools = ['web']

api_version = '2026-08-01-preview'
blob_index_name = f'{prefix}-blob-chunks'  # Use f'{prefix}-cu-chunks' for notebook 05.
sharepoint_index_name = f'{prefix}-nb7-index'
blob_source_name = f'{prefix}-kb-blob'
sharepoint_source_name = f'{prefix}-kb-sharepoint'
webiq_source_name = f'{prefix}-kb-webiq'
knowledge_base_name = f'{prefix}-kb'

credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
print('Search:', search_endpoint)
print('Existing indexes:', blob_index_name, '+', sharepoint_index_name)
print('Knowledge base:', knowledge_base_name)

Search: https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net
Existing indexes: megan01-blob-chunks + megan01-nb7-index
Knowledge base: megan01-kb


## 1. Check the existing indexes

The next cell reads both schemas and checks how many chunks are visible to your signed-in user. Missing indexes or zero visible chunks must be resolved in notebooks 04/05 or 07, not by creating another ingestion pipeline here.

The Blob indexes from 04 and 05 share the same content and source fields, so change only `blob_index_name` above to select between them. The SharePoint source expects notebook 07's Content Understanding index with `UserIds` and `GroupIds` permission fields.

In [2]:
for index_name, lab, semantic_name, content_field, data_fields in [
    (blob_index_name, '04 or 05', 'chunk-semantic', 'chunk',
     ['chunk_id', 'chunk', 'title', 'source/file_name', 'source/url']),
    (sharepoint_index_name, '07', 'semantic-config', 'content',
     ['id', 'content', 'sourceFile', 'webUrl', 'page', 'pageTo']),
]:
    token = search_token()
    headers = {'Authorization': f'Bearer {token}'}
    response = requests.get(
        f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
        headers=headers, timeout=(10, 90),
    )
    if response.status_code == 404:
        raise RuntimeError(f'Index {index_name} not found. Complete notebook {lab} with the same prefix first.')
    response.raise_for_status()
    index = response.json()
    fields = {field['name']: field for field in index['fields']}
    for field in index['fields']:
        for child in field.get('fields', []):
            fields[f"{field['name']}/{child['name']}"] = child
    if (any(name not in fields or fields[name].get('retrievable') is False for name in data_fields)
            or not fields.get(content_field, {}).get('searchable')):
        raise RuntimeError(f'{index_name} does not have the expected retrievable content/source fields. See notebook {lab}.')
    configurations = index.get('semantic', {}).get('configurations', [])
    if semantic_name not in {config['name'] for config in configurations}:
        raise RuntimeError(f'{index_name} is missing semantic configuration {semantic_name}. See notebook {lab}.')
    if index_name == sharepoint_index_name:
        if (index.get('permissionFilterOption') != 'enabled'
                or fields.get('UserIds', {}).get('permissionFilter') != 'userIds'
                or fields.get('GroupIds', {}).get('permissionFilter') != 'groupIds'):
            raise RuntimeError('The SharePoint index must retain notebook 07 user/group permission filtering.')
        headers['x-ms-query-source-authorization'] = token
    response = requests.post(
        f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
        headers=headers, json={'search': '*', 'count': True, 'top': 0}, timeout=(10, 90),
    )
    response.raise_for_status()
    count = response.json()['@odata.count']
    if not count:
        raise RuntimeError(f'No chunks are visible in {index_name}. Check ingestion and document access in notebook {lab}.')
    print(f'{index_name}: {count} visible chunks; existing schema unchanged.')

megan01-blob-chunks: 6 visible chunks; existing schema unchanged.
megan01-nb7-index: 47 visible chunks; existing schema unchanged.


## 2. Reference the Blob index

`searchIndexParameters.searchIndexName` points at the index that already exists. `searchFields` selects its text for retrieval; `sourceDataFields` returns chunk text and source metadata with references. Search uses the existing index's semantic configuration and vectorizer. No storage connection or new embedding deployment is configured here.

In [3]:
blob_source = {
    'name': blob_source_name, 'kind': 'searchIndex',
    'description': 'Travel reimbursement and onboarding policy documents from the existing Blob index.',
    'searchIndexParameters': {
        'searchIndexName': blob_index_name,
        'semanticConfigurationName': 'chunk-semantic',
        'searchFields': [{'name': 'chunk'}],
        'sourceDataFields': [
            {'name': name} for name in ('chunk_id', 'chunk', 'title', 'source/file_name', 'source/url')
        ],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{blob_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=blob_source, timeout=(10, 90),
)
response.raise_for_status()
print('Blob knowledge source ready:', blob_source_name, '->', blob_index_name)

Blob knowledge source ready: megan01-kb-blob -> megan01-blob-chunks


## 3. Reference the SharePoint index

This is also a `searchIndex` knowledge source, **not** `indexedSharePoint` (which would build another ingestion pipeline) or `remoteSharePoint` (which would call Microsoft 365 at query time).

The existing index retains its `permissionFilterOption`, `UserIds` and `GroupIds`. Retrieval sends the signed-in user's Search-scoped token in `x-ms-query-source-authorization`; Search applies the existing ACLs. ACL fields are not returned in source data. Source links and page ranges are returned for citations.

In [4]:
sharepoint_source = {
    'name': sharepoint_source_name, 'kind': 'searchIndex',
    'description': 'Reports and documents from the existing SharePoint library index, with Entra user/group permissions.',
    'searchIndexParameters': {
        'searchIndexName': sharepoint_index_name,
        'semanticConfigurationName': 'semantic-config',
        'searchFields': [{'name': 'content'}],
        'sourceDataFields': [
            {'name': name} for name in ('id', 'content', 'sourceFile', 'webUrl', 'page', 'pageTo')
        ],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{sharepoint_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=sharepoint_source, timeout=(10, 90),
)
response.raise_for_status()
print('SharePoint knowledge source ready:', sharepoint_source_name, '->', sharepoint_index_name)

SharePoint knowledge source ready: megan01-kb-sharepoint -> megan01-nb7-index


## 4. Web IQ knowledge source (MCP server)

Web IQ is a remote knowledge source: Search calls its MCP tools at query time and ranks the results next to your private content. The model in the knowledge base chooses which tool to call and writes its arguments.

**The workshop Web IQ key is stored in this knowledge source** using `storedHeaders` authentication. Search sends `x-apikey` during MCP tool discovery and calls, so the Foundry IQ portal works without supplying custom headers on each question. Search masks stored header values when the source is read back.

Use a workshop-only key: authorized knowledge-base callers share its quota and billing. Restrict source-authoring permissions, rotate/revoke the key after use, and never print the `webiq_source` variable or share outputs containing credentials. The key is loaded from `.env`, not embedded in notebook source. Notebook 10 separately demonstrates keeping a key in APIM instead.

Only the basic **`web` search tool** is configured; the Web IQ Auto tool is not used. `outputParsing: {'kind': 'auto'}` below is Search's MCP response-parsing mode, not a Web IQ tool.

The source description and retrieval instructions tell the planner to use **`contentFormat: passage`**: query-dependent search passages, not full-document text, HTML or Markdown. This works with passage-only profiles; requesting full-document formats can return 403. The tool arguments are visible under `result['activity']` if a call fails.

Preview limits for MCP knowledge sources: `minimal` reasoning effort and `alwaysQuerySource` are not supported, and tool calls can be slow, so the retrieve request sets `maxRuntimeInSeconds`.

In [5]:
webiq_source = {
    'name': webiq_source_name, 'kind': 'mcpServer',
    'description': 'Basic Web IQ web search for public information. Always set contentFormat to passage; do not request full-document text, html or markdown.',
    'resultsProcessing': 'rerank',
    'mcpServerParameters': {
        'serverURL': 'https://api.microsoft.ai/v3/mcp',
        'authentication': {
            'kind': 'storedHeaders',
            'storedHeadersParameters': {'headers': {'x-apikey': webiq_api_key}},
        },
        'tools': [
            {'name': tool, 'outputParsing': {'kind': 'auto'}, 'maxOutputTokens': 2000}
            for tool in webiq_tools
        ],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{webiq_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=webiq_source, timeout=(10, 120),
)
response.raise_for_status()
print('Web IQ knowledge source ready:', webiq_source_name, '| tools:', webiq_tools)

Web IQ knowledge source ready: megan01-kb-webiq | tools: ['web']


## 5. Create the knowledge base

The knowledge base lists the three sources and the model that plans retrieval. For each question the model decides which sources are worth querying, writes a subquery per source, and Search merges and reranks everything into one result. With `answerSynthesis`, the same model then writes a cited answer from the retrieved evidence.

- `retrievalInstructions` tells the planner what each source is for. Clear instructions matter more than anything else here.
- `retrievalReasoningEffort` is `medium`: the Web IQ (MCP) source does not support `minimal`.
- The model is a Foundry deployment that the Search managed identity calls (Cognitive Services User); no model keys are used.

In [ ]:
knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Workshop policies (Blob and SharePoint) plus live public web information from Web IQ.',
    'retrievalInstructions': (
        f'Use {blob_source_name} for travel reimbursement and onboarding policy documents from Blob. '
        f'Use {sharepoint_source_name} for reports and other documents from the SharePoint library. '
        'Consider both indexed sources when an internal question does not identify its document source. '
        f'Use {webiq_source_name} for current public information and external facts. '
        'For every Web IQ web call, explicitly set contentFormat to passage. '
        'Never request text, html or markdown content formats; use only query-dependent search passages. '
        'For internal-versus-public comparisons, query the relevant internal source and Web IQ. '
        'Do not send internal document text to Web IQ.'
    ),
    'answerInstructions': (
        'Answer concisely and cite every source. Say which facts come from internal documents and which '
        'come from the web. If the sources do not contain the answer, say you do not know.'
    ),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [
        {'name': blob_source_name},
        {'name': sharepoint_source_name},
        {'name': webiq_source_name},
    ],
    'models': [{
        'kind': 'azureOpenAI',
        'azureOpenAIParameters': {
            'resourceUri': aoai_endpoint,
            'deploymentId': chat_deployment,
            'modelName': chat_model,
        },
    }],
    'retrievalReasoningEffort': {'kind': 'medium'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()
print('Knowledge base ready:', knowledge_base_name)

Knowledge base ready: megan01-kb


### Use the knowledge base from the Foundry IQ portal

1. Set the workshop `WEBIQ_API_KEY` in your local `.env`; do not paste it into the chat or notebook source.
2. Run **Configuration**, then **4. Web IQ knowledge source** (`create-webiq-source`). Its PUT saves `mcpServerParameters.authentication.kind = storedHeaders` with the `x-apikey` header on `{prefix}-kb-webiq` in Azure AI Search. On a first run, also complete section 5 to create the knowledge base.
3. Open the existing `{prefix}-kb` knowledge base in the Foundry IQ portal. Refresh it and start a new chat after updating authentication. Updating the source does not require recreating the knowledge base or either index.
4. Ask a public-web question such as **What is the latest stable release of Python, and when was it released?** Check that Web IQ activity succeeds and the answer includes references, then try the internal/web comparison below.

**Why this is needed:** the portal does not supply the notebook's former per-request Web IQ control headers. Without source authentication, MCP tool discovery returns **401 `auth_missing_token`** even when notebook requests that supply a key succeed. A model saying it cannot browse is not the diagnosis; check the retrieval activity/error.

The credential lives on the **Search knowledge source**, not in the portal chat. Search masks it on readback and supplies it to Web IQ automatically. Source callers share its quota/billing, but each user's SharePoint document access is still enforced separately. To rotate the key, update `.env` and rerun Configuration plus `create-webiq-source`.

See [MCP knowledge-source authentication options](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server#authentication-options).

## 6. Ask questions

Each request carries your Search identity:

- `Authorization`: your token for Azure AI Search.
- `x-ms-query-source-authorization`: your token again, so SharePoint permissions are evaluated as **you**, not as the service.

Search supplies the stored Web IQ key from the knowledge source. Retrieve requests need no Web IQ control headers; the same source can therefore be queried from the Foundry IQ portal. Refresh the portal after creating or updating the source, then start a new chat to avoid retaining an earlier failed answer.

`includeActivity` returns what each source did, so you can see which sources the planner actually used. The questions name sources to make this first check explicit; source selection is still the model's judgment, not a guarantee. After verifying each source, try questions without source names.

Change these example questions to match your indexed documents. A source may be skipped by the planner; inspect activity and references rather than assuming all three sources were used.

### 6a. An internal question

In [7]:
question = f'Using {blob_source_name}, what receipts do I need to submit for travel reimbursement?'
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
if response.status_code >= 400:
    print('Retrieve failed:', response.status_code, '| request:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
    print(response.text[:8000])
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

From the internal travel policy document:  
- Submit your reimbursement within 30 calendar days after the trip ends [ref_id:0].  
- Attach an itemized receipt for lodging [ref_id:0].  
- Attach an itemized receipt for transport [ref_id:0].  
- Attach an itemized receipt for any individual expense greater than 25 (fictional) credits [ref_id:0].  
- A card statement by itself is not acceptable as an itemized receipt [ref_id:0].

References:
- searchIndex 0 a1651ecf03a2_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDEvYmxvYi90cmF2ZWwtcG9saWN5LnR4dA2_chunks_0
- searchIndex 1 a1651ecf03a2_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL21lZ2FuMDEvYmxvYi90cmF2ZWwtcG9saWN5LnR4dA2_chunks_2

Activity:
- modelQueryPlanning   
- searchIndex megan01-kb-blob 2 
- modelAnswerSynthesis   
- agenticReasoning   


### 6b. A question only the web can answer

Internal documents cannot answer this. A good run shows the Web IQ source used and the internal sources skipped or returning nothing relevant.

In [25]:
question = 'What is the weather in MN?'
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
if response.status_code >= 400:
    print('Retrieve failed:', response.status_code, '| request:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
    print(response.text[:8000])
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

Do you mean the whole state of Minnesota or a specific city (e.g., Minneapolis, St. Paul, Duluth, Rochester)? Which location should I check the current weather for?

References:

Activity:
- modelAnswerSynthesis   
- agenticReasoning   


In [14]:
print("HTTP status:", response.status_code)
print(json.dumps(result.get("activity", []), indent=2))

print(json.dumps(
    [s for s in result.get("activity", []) if s.get("type") == "mcpServer"],
    indent=2,
))
print(json.dumps(result.get("references", []), indent=2))

HTTP status: 200
[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "startedAt": "2026-11-13T20:30:16.2910986Z",
    "completedAt": "2026-11-13T20:30:17.6845292Z",
    "inputTokens": 2313,
    "outputTokens": 69,
    "elapsedMs": 1393,
    "model": {
      "modelName": "gpt-5-mini",
      "deploymentId": "workshop-model"
    }
  },
  {
    "type": "mcpServer",
    "id": 1,
    "startedAt": "2026-11-13T20:30:17.684797Z",
    "completedAt": "2026-11-13T20:30:17.9262774Z",
    "knowledgeSourceName": "megan01-kb-webiq",
    "queryTime": "2026-11-13T20:30:17.6849041Z",
    "count": 1,
    "elapsedMs": 167,
    "mcpServerArguments": {
      "toolName": "web",
      "toolArguments": {
        "query": "latest stable release of Python release date \"latest stable\" Python",
        "maxResults": 10,
        "contentFormat": "passage"
      }
    }
  },
  {
    "type": "modelAnswerSynthesis",
    "id": 2,
    "startedAt": "2026-11-13T20:30:17.9284337Z",
    "completedAt": "2026-11-13T20:3

### 6c. A question for the indexed SharePoint documents

Change the topic to match your SharePoint documents. This example uses sustainability and AI; it does not assume the SharePoint library contains the Blob travel policy. Verify that SharePoint activity returns chunks and references, not just HTTP 200.

In [ ]:
question = f'Using {sharepoint_source_name}, what are the main recommendations about sustainability and AI?'
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
if response.status_code >= 400:
    print('Retrieve failed:', response.status_code, '| request:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
    print(response.text[:8000])
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

### 6d. A question that needs internal content and the web

This is the point of one knowledge base: the planner should query internal documents **and** the web, and the answer should separate the two.

In [ ]:
question = (f'Using {blob_source_name} and {webiq_source_name}, how does our travel reimbursement policy '
            'compare with common corporate travel expense best practices on the public web?')
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 90,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': 'searchIndex',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 150),
)
if response.status_code >= 400:
    print('Retrieve failed:', response.status_code, '| request:', response.headers.get('x-ms-request-id') or response.headers.get('request-id'))
    print(response.text[:8000])
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

## 7. Evaluate, do not just read the answer

- A fluent answer is not proof. Open `references` and confirm each claim traces to a Blob document, SharePoint item or web result. Source data contains the existing index's metadata, including SharePoint URLs and page ranges.
- Ask a question matching your SharePoint documents and confirm that `{prefix}-kb-sharepoint` appears in the activity with results. A Blob-only success does not prove SharePoint retrieval.
- Ask the same SharePoint question as two users with different document access. The existing index enforces its Entra ACLs; this notebook does not change or expand document permissions.
- Blob content has no per-user document ACLs in these workshop indexes: Search data access governs who can query it.
- Compare activity for 6a-6d. Source selection is model-driven. Adjust source descriptions and `retrievalInstructions` in section 5 if routing does not match your content.

## 8. Optional cleanup

Resources are retained by default for notebooks 10 and 12. The opt-in cell deletes only this notebook's knowledge base and its three knowledge-source definitions. It **never deletes the existing indexes or their ingestion resources**, Blob files or SharePoint documents.

A knowledge source cannot be deleted while another knowledge base references it. Remove those dependent knowledge bases first if applicable. Clear notebook outputs before sharing; they can contain internal document text and web results.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [
        ('knowledgebases', knowledge_base_name), ('knowledgesources', webiq_source_name),
        ('knowledgesources', sharepoint_source_name), ('knowledgesources', blob_source_name),
    ]:
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', name)
else:
    print('Cleanup disabled; existing indexes and knowledge resources retained.')

## References

- [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq)
- [Existing Search index knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-search-index)
- [Permission filtering at query time](https://learn.microsoft.com/azure/search/search-query-access-control-rbac-enforcement)
- [MCP server knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve from a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

Preview APIs and model availability can change.